In [0]:
%pip install databricks_vectorsearch 
dbutils.library.restartPython()

In [0]:
from config import DeployConfig
import requests
import json

In [0]:
dbutils.widgets.text("config_path", "./config/env_variables.yml")
config_path = dbutils.widgets.get("config_path")
cfg = DeployConfig.from_yaml(config_path)

In [0]:
vs_index = getattr(cfg, f"vs_index")
image_table = getattr(cfg, f"image_table")

In [0]:
vs_index.path

#AGENT BUILD

In [0]:
import mlflow
from mlflow.deployments import get_deploy_client
from databricks.vector_search.index import VectorSearchIndex
from databricks.vector_search.client import VectorSearchClient
from mlflow.entities import SpanType
from PIL import Image
from io import BytesIO

In [0]:
class CAT_AGENT(mlflow.pyfunc.PythonModel):
    def __init__(self):
      self.client = get_deploy_client("databricks")
      self.vsc=VectorSearchClient() #will add oauth to make faster here

    def load_context(self, context):
      from transformers import CLIPProcessor, CLIPModel
      from PIL import Image
      from io import BytesIO
      # Initialize tokenizer and model
      # will not need to do this once ai_query can take params of a pyfunc. We can have teh serving endpoint be able to do image and text embeddings. 
      self.model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
      self.processor= CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

    @mlflow.trace(name="compute_text_embedding", span_type=SpanType.EMBEDDING, attributes={"model": "clip-vit-large-patch14"})
    def _get_text_embedding(self, text):
      """
      computes the text embedding for a given text.
      """
      #want to change this to call the serving endpoint when ai_query can take params of a pyfunc.
      inputs = self.processor(text=text, return_tensors="pt", padding=True)
      text_features = self.model.get_text_features(**inputs)
      return text_features.detach().numpy().tolist()[0]
    
    @mlflow.trace(name="cat_image_vs_imageemb_lookup", span_type=SpanType.RETRIEVER, attributes={"model": "clip-vit-large-patch14", "vs_index": vs_index.path})
    def _vector_search_retrieval(self, query):
      text_embed_query=self._get_text_embedding(query)
      index = self.vsc.get_index(endpoint_name=vs_index.endpoint, index_name=vs_index.path)
      vs_output = index.similarity_search(columns=["id", 'path'], query_vector=text_embed_query, num_results=3)
      return vs_output
    
    @mlflow.trace(name="cat_id_image_lookup", span_type=SpanType.RETRIEVER, attributes={"table": image_table.path})
    def _get_images(self, vs_output):
      """
      takes the output of vs and querys original delta table for images. 
      uses PIL to open images?
      """
      image_ids=[result[0] for result in vs_output['result']['data_array']]
      image_lookups=spark.sql(f'select content from {image_table.path} where id in ({",".join([str(id) for id in image_ids])})').collect()
      # return Image.open(BytesIO(image_lookups[0]['content']))
      for image in image_lookups:
        yield Image.open(BytesIO(image['content']))

    @mlflow.trace(name="quickstart-agent")
    def predict(self, context, model_input):
      vs_output=self._vector_search_retrieval(query=model_input)
      cats=self._get_images(vs_output)
      return cats



In [0]:
agent=CAT_AGENT()
agent.load_context(None)

In [0]:
# import mlflow
# from mlflow.deployments import get_deploy_client


# class QAChain(mlflow.pyfunc.PythonModel):
#     def __init__(self):
#         self.client = get_deploy_client("databricks")

#     @mlflow.trace(name="quickstart-chain")
#     def predict(self, model_input, system_prompt, params):
#         messages = [
#                 {
#                     "role": "system",
#                     "content": system_prompt,
#                 },
#                 {
#                     "role": "user",
#                     "content":  model_input[0]["query"]
#                 }
#           ]
        
#         traced_predict = mlflow.trace(self.client.predict)
#         output = traced_predict(
#             endpoint=params["model_name"],
#             inputs={
#                 "temperature": params["temperature"],
#                 "max_tokens": params["max_tokens"],
#                 "messages": messages,
#             },
#         )
        
#         with mlflow.start_span(name="_final_answer") as span:
#             span.set_inputs({"query": model_input[0]["query"]})

#             answer = output["choices"][0]["message"]["content"]

#             span.set_outputs({"generated_text": answer})     
#      # Attributes computed at runtime can be set using the set_attributes() method.
#      span.set_attributes({
#        "model_name": params["model_name"],
#                 "prompt_tokens": output["usage"]["prompt_tokens"],
#                 "completion_tokens": output["usage"]["completion_tokens"],
#                 "total_tokens": output["usage"]["total_tokens"]
#             })
#         return answer

In [0]:
response=agent.predict(model_input='orange tabby cat', context=None)

In [0]:
next(response)

In [0]:
next(response)

In [0]:
next(response)